<a href="https://colab.research.google.com/github/s-chudmunge/Notebooks/blob/main/Name_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Environment Check
We begin by checking the GPU status using `nvidia-smi`.

In [2]:
!nvidia-smi

Sun Aug  9 03:58:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Dependencies
We install the necessary libraries for fine-tuning, including `transformers`, `peft`, and `trl`.

In [3]:
!pip -q install -U transformers datasets peft trl bitsandbytes accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.3 MB/s eta 0:00:00


### Library Imports
Importing core libraries and checking versions to ensure environment stability.

In [ ]:
import torch
import transformers
import datasets
import peft
import trl

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

### Model Initialization
Loading the base Qwen2.5 model with 4-bit quantization config to save memory.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Loaded!")

### Storage Setup
Mounting Google Drive to access our training dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Data Discovery
Locating the JSONL dataset file within the mounted drive.

In [ ]:
!find /content/drive -name "master_training_dataset.jsonl"

### Schema Analysis
Scanning the dataset to identify all available fields and data types.

In [ ]:
import json

fields = {}

with open("/content/drive/MyDrive/colab_data/master_training_dataset.jsonl") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)

        def scan(d, prefix=""):
            for k, v in d.items():
                name = f"{prefix}.{k}" if prefix else k
                if isinstance(v, dict):
                    scan(v, name)
                else:
                    t = type(v).__name__
                    fields.setdefault(name, set()).add(t)

        scan(obj)

print(fields)

### Dataset Loading
Loading the data into the Hugging Face `datasets` format with a defined schema.

In [ ]:
from datasets import load_dataset, Features, Value

features = Features({
    "content": Value("string"),
    "title": Value("string"),
    "metrics": {
        "impressions": Value("int64"),
        "clicks": Value("int64"),
        "ctr": Value("float64"),
        "claps": Value("int64"),
        "responses": Value("int64"),
        "score": Value("int64"),
        "comments": Value("int64"),
        "views": Value("int64"),
        "likes": Value("int64"),
        "shares": Value("int64"),
        "facebook": Value("int64"),
        "linkedin": Value("int64"),
    },
    "metadata": {
        "platform": Value("string"),
        "test_id": Value("string"),
        "is_winner": Value("bool"),
        "video_id": Value("string"),
        "channel": Value("string"),
        "topic": Value("string"),
        "source": Value("string"),
    },
    "reward_score": Value("float64"),
})

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/colab_data/master_training_dataset.jsonl",
    features=features,
    split="train",
)

print(dataset)
print(dataset[0])

### Data Splitting
Dividing the dataset into training and validation sets (90/10 split).

In [ ]:
from datasets import DatasetDict

# Train / Validation split
splits = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_ds = splits["train"]
val_ds = splits["test"]

print(train_ds)
print(val_ds)

### Prompt Formatting
Applying a chat template to convert raw content and titles into instructions for the LLM.

In [ ]:
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are an expert at writing highly engaging titles."
        },
        {
            "role": "user",
            "content": f"Generate a high-engagement title for the following content:\n\n{example['content']}"
        },
        {
            "role": "assistant",
            "content": example["title"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        ),
        "reward_score": example["reward_score"]
    }

train_ds = train_ds.map(
    format_example,
    remove_columns=[
        "content",
        "title",
        "metrics",
        "metadata",
    ],
    desc="Formatting train"
)

val_ds = val_ds.map(
    format_example,
    remove_columns=[
        "content",
        "title",
        "metrics",
        "metadata",
    ],
    desc="Formatting validation"
)

print(train_ds.features)
print(train_ds[0])

### Sequence Analysis
Analyzing the token length distribution of our training examples.

# Can we train the model to prefer high-engagement titles?

Standard training learns the *average* style of all titles. To optimize for engagement, we have three main choices:
1. **Filter:** Train only on the best titles (loses a lot of data).
2. **RLHF/DPO:** Advanced methods that compare good vs. bad titles (more complex).
3. **Weighted Training (Our Choice):** We keep all data but tell the model to pay more attention to titles with high `reward_score` values.

**Why Weighted SFT?**
It preserves all your data while ensuring a title with a 0.95 score influences the model much more than one with 0.10.

**The Implementation Plan:**
Since standard tools don't support "weighted loss" by default, we will:
1. **Keep** the `reward_score` in our dataset.
2. **Tokenize** the text for the model.
3. **Customize the Trainer:** Create a `WeightedTrainer` class that overrides the loss calculation.
4. **Apply Weights:** Multiply the error (loss) of each example by its `reward_score` so the model learns more from high-performing titles.

In [ ]:
lengths = []

for ex in train_ds.select(range(10000)):
    lengths.append(len(tokenizer(ex["text"]).input_ids))

import numpy as np

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("95%:", np.percentile(lengths, 95))
print("99%:", np.percentile(lengths, 99))
print("Max:", np.max(lengths))

### Tokenization
Converting the formatted text into input IDs and preparing the labels and rewards for the model.

In [ ]:
MAX_LENGTH = 512

def tokenize(example):
    enc = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    enc["labels"] = enc["input_ids"].copy()
    enc["reward_score"] = example["reward_score"]

    return enc

train_tok = train_ds.map(
    tokenize,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train"
)

val_tok = val_ds.map(
    tokenize,
    remove_columns=val_ds.column_names,
    desc="Tokenizing validation"
)

print(train_tok.features)
print(train_tok[0].keys())

### Data Collation
Setting up the data collator to handle batching and padding during training.

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

### Custom Loss Implementation
Defining a `WeightedTrainer` that scales loss based on the `reward_score` of each example.

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        reward = inputs.pop("reward_score").float()
        weights = 1.0 + reward

        labels = inputs["labels"]

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
        )

        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(
            reduction="none",
            ignore_index=-100,
        )

        token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )

        token_loss = token_loss.view(
            shift_labels.size(0),
            shift_labels.size(1),
        )

        valid_tokens = (shift_labels != -100).float()

        seq_loss = (
            (token_loss * valid_tokens).sum(dim=1)
            / valid_tokens.sum(dim=1).clamp(min=1)
        )

        loss = (seq_loss * weights).sum() / weights.sum()

        return (loss, outputs) if return_outputs else loss

### PEFT Configuration
Applying LoRA (Low-Rank Adaptation) to the model to allow efficient fine-tuning.

# **TRAINING ARGUMENTS**  ⚖

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/title_model",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,

    logging_steps=20,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Model Type Verification
Confirming that the model has been correctly wrapped as a `PeftModel`.

In [ ]:
from peft import PeftModel

print(isinstance(model, PeftModel))

In [ ]:
from peft import PeftModel

print(type(model))
print(isinstance(model, PeftModel))

### Execution
Launching the training process on a subset of the data.

# **TRAINING** ✅

In [ ]:
# Same 25k training examples
train_25k = train_tok.shuffle(seed=42).select(range(25000))

# Random 5k validation examples
val_5k = val_tok.shuffle(seed=42).select(range(5000))

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_25k,
    eval_dataset=val_5k,
    data_collator=data_collator,
)

trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/colab_data/title_model/checkpoint-3000"
)

### Persistence
Saving the fine-tuned LoRA weights and tokenizer locally.

In [ ]:
model.save_pretrained("/content/drive/MyDrive/title_model/final_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/title_model/final_lora")

### Inference
Generating a title using the fine-tuned model to test its performance.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an expert at writing highly engaging titles."
    },
    {
        "role": "user",
        "content": """ListenThe landscape of sequence modeling was once defined by the sequential nature of Recurrent Neural Networks (RNNs) and the local receptive fields of Convolutional Neural Networks (CNNs). "Attention Is All You Need" fundamentally disrupted this history by proving that recurrence and convolution are entirely unnecessary for state-of-the-art sequence modeling. By introducing the Transformer architecture, the authors demonstrated that a purely attention-based mechanism can capture global dependencies in parallel, paving the way for the era of Large Language Models and foundational AI.#The Renaissance of Global AttentionPrior to the Transformer, attention was used primarily as an augmentation for RNNs, helping the model "focus" on specific parts of an input sequence during decoding. The Transformer elevated attention from a supporting component to the primary architectural primitive. By eliminating sequential processing, the architecture allowed for unprecedented parallelization during training, enabling models to ingest massive datasets with a constant computational depth. This shift effectively solved the "vanishing gradient" problem inherent in long RNN sequences, as every token in a Transformer has a direct path to every other token, regardless of their distance in the sequence.#Scaled Dot-Product Attention: The Mathematical StabilizerAt the core of the Transformer is the Scaled Dot-Product Attention mechanism. It computes a relationship between a Query (QQ), a Key (KK), and a Value (VV) by calculating the dot product of QQ and KK, scaling the result, and applying a softmax function to weight the values in VV. The critical innovation here is the scaling factor of 1/dk1/\sqrt{d_k}.
The authors observed that as the dimensionality of the keys (dkd_k) increases, the magnitude of the dot products grows, pushing the softmax function into regions where the gradient is extremely small. By dividing the dot product by the square root of the dimension, the model preserves a unit variance, ensuring that gradients remain stable during backpropagation. This mathematical stabilizer is what allows Transformers to scale to the massive hidden dimensions seen in modern architectures.#Multi-Head Attention: Parallelizing Subspace ReasoningRather than performing a single attention operation across the entire hidden dimension, the Transformer splits the model's representation into multiple "heads." Each head performs an independent attention operation in a unique subspace. This allows the model to "jointly attend" to information from different perspectives simultaneously. One head might learn to identify syntactic relationships (e.g., subject-verb agreement), while another focuses on semantic resolution (e.g., pronoun antecedents). By concatenating the outputs of these heads and projecting them back into the model dimension, the Transformer achieves a high-density reasoning capability that a single attention head could not replicate.#The Residual Encoder-Decoder StackThe Transformer architecture is composed of an Encoder and a Decoder, each consisting of a stack of N=6N=6 identical layers. The Encoder generates a continuous representation of the input, while the Decoder utilizes that representation to generate an output sequence one token at a time. Every sub-layer within these blocks - whether it is an attention mechanism or a feed-forward network - is wrapped in a residual connection followed by Layer Normalization. This "Add & Norm" pattern is vital for deep scaling, as it allows gradients to flow through the network without degradation, maintaining the structural integrity of the representations as they pass through dozens of layers.#Position-Wise Feed-Forward NetworksEach layer in the Transformer stack contains a fully connected Feed-Forward Network (FFN) that is applied to each position separately and identically. This FFN consists of two linear transformations with a ReLU activation in between. While the attention mechanism is responsible for moving information between positions, the FFN is where the "heavy lifting" of data transformation occurs at each individual position. In the original model, the inner dimension of the FFN was four times larger than the model dimension (20482048 vs 512512), providing the necessary computational capacity for the model to refine and process the relational data captured by the attention heads.#Sinusoidal Positional EncodingBecause the Transformer contains no recurrence or convolution, it is inherently permutation-invariant - it treats the input as a "bag of tokens" with no sense of order. To inject positional information, the authors introduced Sinusoidal Positional Encodings. These are added to the input embeddings and use a series of sine and cosine functions of different frequencies to encode the absolute position of each token. The choice of sinusoids was deliberate: the authors hypothesized that it would allow the model to learn to attend by relative positions, as any fixed offset can be represented as a linear function of the current position. This also allows the model to theoretically generalize to sequence lengths longer than those encountered during training.#Masking and Teacher ForcingThe training of the Transformer employs a technique called Teacher Forcing, where the Decoder is provided with the ground-truth previous tokens to predict the next one. To prevent the model from "cheating" by looking ahead at the target it is trying to predict, the Decoder uses Masked Multi-Head Attention. This masking sets the attention scores for future positions to −∞-\infty before the softmax step, ensuring that the prediction for a specific token can only depend on the tokens that preceded it. This causal constraint is what allows Transformers to function as effective autoregressive generators.#The Legacy of the TransformerThe Transformer did more than just improve translation scores; it provided the blueprint for the unification of machine learning. The same architectural primitives now power vision models (ViT), audio systems, and even robotics. By proving that self-attention is a universal sequence processor, the paper set the stage for the convergence of AI research, where the focus shifted from hand-engineered architectural priors to the pure scaling of attention-based blocks.Join the EulerFold communityTrack progress and collaborate on courses with students worldwide.Create your step by step course #Dive DeeperAttention Is All You Need (Original Paper)arXiv • articleExplore Resource The Illustrated Transformer (Jay Alammar)Blog • articleExplore Resource The Annotated Transformer (Harvard NLP)Blog • articleExplore Resource Attention Is All You Need: WalkthroughYouTube • videoExplore Resource Discussion0Join the discussionSign in to share your thoughts and technical insights. Sign In FreeNo discussions yet. Be the first to share an insight.Recommended ReadingsFrom the GlossaryWhy AI Models Get Lost in Long DocumentsResearch DecodedThe New Architecture Challenging TransformersThe Hardware Trick That Sped Up TransformersWhy Transformers Are Replacing Traditional VisionThe author of this article utilized generative AI (Google Gemini 3.1 Pro) to assist in part of the drafting and editing process. Previous DecodingWhy We Can Finally Train 100-Layer NetworksHe, Zhang, Ren, Sun (Microsoft Research, 2015)Next Decoding The Predictable Intelligence of ScalingKaplan et al. (OpenAI, 2020)"""
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

In [ ]:
text = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = base_tokenizer(text, return_tensors="pt").to(base_model.device)

output = base_model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
)

print(base_tokenizer.decode(output[0], skip_special_tokens=True))